# Project: Monte Carlo Simulations with NumPy

Use randomness to estimate pi, simulate stock prices, and analyze probability distributions with NumPy.

In [ ]:
import sys, os, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_PATH = '/content/drive/MyDrive/data-science-mastery'
    if os.path.isdir(REPO_PATH):
        os.chdir(REPO_PATH)
        print(f'Working directory: {os.getcwd()}')
        if not os.path.isdir('Data') or len(os.listdir('Data')) < 5:
            subprocess.check_call([sys.executable, 'scripts/prepare_datasets.py'])
    else:
        REPO_PATH = '/content/data-science-mastery'
        if os.path.isdir(REPO_PATH):
            os.chdir(REPO_PATH)
        else:
            print('Repo not found. Run opencolab_setup.ipynb first.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
np.random.seed(42)
print('NumPy version:', np.__version__)

## 1. Estimating Pi with Random Points

In [ ]:
def estimate_pi(n_points):
    x = np.random.uniform(0, 1, n_points)
    y = np.random.uniform(0, 1, n_points)
    inside = x**2 + y**2 <= 1
    pi_est = 4 * np.sum(inside) / n_points
    return pi_est, abs(pi_est - np.pi), inside, x, y

n_small = 5000
pi_est, err, inside, x, y = estimate_pi(n_small)
print(f'n={n_small}: pi = {pi_est:.6f} (error={err:.6f})')
plt.figure(figsize=(6, 6))
plt.scatter(x[inside], y[inside], c='steelblue', s=1, alpha=0.5, label='Inside')
plt.scatter(x[~inside], y[~inside], c='coral', s=1, alpha=0.5, label='Outside')
plt.legend(markerscale=10)
plt.title(f'Estimating Pi: n={n_small}')
plt.axis('equal')
plt.show()

In [ ]:
sizes = np.logspace(2, 6, 20, dtype=int)
pi_estimates = np.array([estimate_pi(n)[0] for n in sizes])
plt.figure(figsize=(10, 5))
plt.semilogx(sizes, pi_estimates, 'o-', label='Estimate')
plt.axhline(y=np.pi, color='r', linestyle='--', label=f'True pi = {np.pi:.6f}')
plt.xlabel('Number of points')
plt.ylabel('Estimated pi')
plt.title('Convergence of Monte Carlo Pi Estimation')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
print(f'Final: n={sizes[-1]}: pi = {pi_estimates[-1]:.6f}')

## 2. Stock Price Simulation (Random Walk)

In [ ]:
def simulate_stock(S0=100, mu=0.05, sigma=0.2, T=1, dt=1/252, n_sims=1000):
    n_steps = int(T / dt)
    t = np.linspace(0, T, n_steps)
    epsilon = np.random.normal(0, 1, (n_sims, n_steps))
    drift = (mu - 0.5 * sigma**2) * dt
    diffusion = sigma * np.sqrt(dt) * epsilon
    log_returns = np.cumsum(drift + diffusion, axis=1)
    prices = S0 * np.exp(log_returns)
    return t, prices

t, prices = simulate_stock(n_sims=500)
plt.figure(figsize=(12, 6))
plt.plot(t * 252, prices[:50].T, alpha=0.3, color='steelblue')
mean_price = np.mean(prices, axis=0)
plt.plot(t * 252, mean_price, 'r-', linewidth=3, label='Mean')
plt.xlabel('Trading Days')
plt.ylabel('Price ($)')
plt.title('Monte Carlo Stock Price Simulations (GBM)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
final_prices = prices[:, -1]
print(f'Expected price: ${np.mean(final_prices):.2f}')
print(f'95% CI: (${np.percentile(final_prices, 2.5):.2f}, ${np.percentile(final_prices, 97.5):.2f})')

## 3. The Birthday Problem

In [ ]:
def birthday_simulation(n_people, n_sims=10000):
    birthdays = np.random.randint(0, 365, (n_sims, n_people))
    has_dup = np.array([len(set(b)) < len(b) for b in birthdays])
    return np.mean(has_dup)

group_sizes = np.arange(5, 61, 5)
probs = [birthday_simulation(n, 5000) for n in group_sizes]
plt.figure(figsize=(10, 5))
plt.plot(group_sizes, probs, 'o-', linewidth=2, markersize=8)
plt.axhline(y=0.5, color='r', linestyle='--', alpha=0.7, label='50%')
plt.axvline(x=23, color='g', linestyle='--', alpha=0.7, label='n=23')
plt.xlabel('Number of People')
plt.ylabel('Probability of Shared Birthday')
plt.title('Birthday Problem: Monte Carlo')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
print(f'With 23 people: P(shared birthday) = {birthday_simulation(23, 10000):.3f}')

## 4. Variance Reduction: Antithetic Variates

In [ ]:
def mc_option_price(S0=100, K=105, T=1, r=0.05, sigma=0.2, n_sims=100000):
    Z = np.random.normal(0, 1, n_sims // 2)
    ST1 = S0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    ST2 = S0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*(-Z))
    payoff = 0.5 * (np.maximum(ST1 - K, 0) + np.maximum(ST2 - K, 0))
    return np.exp(-r*T) * np.mean(payoff)

price = mc_option_price(n_sims=50000)
print(f'Call option price (antithetic): ${price:.4f}')

## Summary

- Random sampling for pi estimation
- Stock price simulation with geometric Brownian motion
- Birthday problem solved with Monte Carlo
- Variance reduction with antithetic variates